In [2]:
import pickle
import streamlit as st

In [3]:
import os
os.chdir("C:/Users/Simon/Documents/UZH/DataAnalysiswithPython/SimonteBiesebeekPlayground/SimonteBiesebeekPlayground/FinalProject")
print(os.getcwd())

C:\Users\Simon\Documents\UZH\DataAnalysiswithPython\SimonteBiesebeekPlayground\SimonteBiesebeekPlayground\FinalProject


In [4]:
import os
import pandas as pd

csv_files = [f for f in os.listdir("PremierData") if f.endswith(".csv")]
dataframes = {f: pd.read_csv(os.path.join("PremierData", f)) for f in csv_files}

In [5]:
files = os.listdir("PremierData")
print(files)

['PremierData2021.csv', 'PremierData2122.csv', 'PremierData2223.csv', 'PremierData2324.csv', 'PremierData2425.csv']


In [6]:
from pathlib import Path

data_folder = Path("PremierData")

# Create one combined DataFrame, adding a Season column automatically
premier_data = pd.concat(
    [
        pd.read_csv(f).assign(
            Season=f.stem.replace("PremierData", "")  # e.g. "Data2122" → "2122"
        )
        for f in data_folder.glob("*.csv")
    ],
    ignore_index=True
)
premier_data.to_csv("MergedPremierLeagueData.csv", index=False)
merged_premier = pd.read_csv("MergedPremierLeagueData.csv")
merged_premier['Date'] = pd.to_datetime(merged_premier['Date'], dayfirst=True, errors='coerce')
merged_premier['Time'] = pd.to_datetime(merged_premier['Time'], format='%H:%M', errors='coerce').dt.time

# 2. Sort chronologically
merged_premier = merged_premier.sort_values(by=['Date', 'Time']).reset_index(drop=True)

# 3. Reformat them for display (DD/MM/YYYY and HH:MM)
merged_premier['Date'] = merged_premier['Date'].dt.strftime('%d/%m/%Y')
merged_premier['Time'] = merged_premier['Time'].apply(lambda t: t.strftime('%H:%M') if pd.notnull(t) else '')

# 4. Preview
merged_premier[['Date', 'Time']].head()
merged_premier.head()


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,1XBCH,1XBCD,1XBCA,BFECH,BFECD,BFECA,BFEC>2.5,BFEC<2.5,BFECAHH,BFECAHA
0,E0,12/09/2020,12:30,Fulham,Arsenal,0,3,A,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E0,12/09/2020,15:00,Crystal Palace,Southampton,1,0,H,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E0,12/09/2020,17:30,Liverpool,Leeds,4,3,H,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E0,12/09/2020,20:00,West Ham,Newcastle,0,2,A,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E0,13/09/2020,14:00,West Brom,Leicester,0,3,A,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
print(merged_premier.shape)

(1900, 133)


In [8]:
merged_premier['Winner'] = merged_premier.apply(
    lambda row: row['HomeTeam'] if row['FTR'] == 'H' 
                else (row['AwayTeam'] if row['FTR'] == 'A' else 'Draw'),
    axis=1
)
print(merged_premier.head())

  Div        Date   Time        HomeTeam     AwayTeam  FTHG  FTAG FTR  HTHG  \
0  E0  12/09/2020  12:30          Fulham      Arsenal     0     3   A     0   
1  E0  12/09/2020  15:00  Crystal Palace  Southampton     1     0   H     1   
2  E0  12/09/2020  17:30       Liverpool        Leeds     4     3   H     3   
3  E0  12/09/2020  20:00        West Ham    Newcastle     0     2   A     0   
4  E0  13/09/2020  14:00       West Brom    Leicester     0     3   A     0   

   HTAG  ... 1XBCD 1XBCA  BFECH  BFECD  BFECA  BFEC>2.5  BFEC<2.5  BFECAHH  \
0     1  ...   NaN   NaN    NaN    NaN    NaN       NaN       NaN      NaN   
1     0  ...   NaN   NaN    NaN    NaN    NaN       NaN       NaN      NaN   
2     2  ...   NaN   NaN    NaN    NaN    NaN       NaN       NaN      NaN   
3     0  ...   NaN   NaN    NaN    NaN    NaN       NaN       NaN      NaN   
4     0  ...   NaN   NaN    NaN    NaN    NaN       NaN       NaN      NaN   

   BFECAHA          Winner  
0      NaN         Arsenal 

In [9]:
merged_premier['HomeTeam'].unique()

array(['Fulham', 'Crystal Palace', 'Liverpool', 'West Ham', 'West Brom',
       'Tottenham', 'Sheffield United', 'Brighton', 'Everton', 'Leeds',
       'Man United', 'Arsenal', 'Southampton', 'Newcastle', 'Chelsea',
       'Leicester', 'Aston Villa', 'Wolves', 'Burnley', 'Man City',
       'Brentford', 'Watford', 'Norwich', 'Bournemouth', "Nott'm Forest",
       'Luton', 'Ipswich'], dtype=object)

In [10]:
merged_premier["GoalDifference"] = merged_premier["FTHG"] - merged_premier["FTAG"]
print(merged_premier["GoalDifference"])

0      -3
1       1
2       1
3      -2
4      -3
       ..
1895   -1
1896   -1
1897   -1
1898   -3
1899    0
Name: GoalDifference, Length: 1900, dtype: int64


In [11]:
merged_premier['FullTimeScore'] = merged_premier['FTHG'].astype(str) + '-' + merged_premier['FTAG'].astype(str)
print(merged_premier[['FTHG', 'FTAG', 'FullTimeScore']].head())

   FTHG  FTAG FullTimeScore
0     0     3           0-3
1     1     0           1-0
2     4     3           4-3
3     0     2           0-2
4     0     3           0-3


In [12]:
import numpy as np

# Assign points based on match result (FTR = 'H', 'A', 'D')
merged_premier["HomePoints"] = np.select(
    [
        merged_premier["FTR"] == "H",  # Home win
        merged_premier["FTR"] == "D",  # Draw
        merged_premier["FTR"] == "A"   # Away win
    ],
    [3, 1, 0]
)

merged_premier["AwayPoints"] = np.select(
    [
        merged_premier["FTR"] == "A",  # Away win
        merged_premier["FTR"] == "D",  # Draw
        merged_premier["FTR"] == "H"   # Home win
    ],
    [3, 1, 0]
)

print(merged_premier[["FTR", "HomePoints", "AwayPoints"]])

     FTR  HomePoints  AwayPoints
0      A           0           3
1      H           3           0
2      H           3           0
3      A           0           3
4      A           0           3
...   ..         ...         ...
1895   A           0           3
1896   A           0           3
1897   A           0           3
1898   A           0           3
1899   D           1           1

[1900 rows x 3 columns]


In [13]:
home_points = merged_premier.groupby("HomeTeam")["HomePoints"].sum().reset_index()
away_points = merged_premier.groupby("AwayTeam")["AwayPoints"].sum().reset_index()
total_points = pd.merge(home_points, away_points, left_on="HomeTeam", right_on="AwayTeam", how="outer").fillna(0)
total_points["Team"] = total_points["HomeTeam"].combine_first(total_points["AwayTeam"])
print(total_points.head())

      HomeTeam  HomePoints     AwayTeam  AwayPoints         Team
0      Arsenal         200      Arsenal         177      Arsenal
1  Aston Villa         166  Aston Villa         129  Aston Villa
2  Bournemouth          77  Bournemouth          66  Bournemouth
3    Brentford         114    Brentford          86    Brentford
4     Brighton         139     Brighton         124     Brighton


In [14]:
home_gd = merged_premier.groupby("HomeTeam")["GoalDifference"].sum().reset_index()
home_gd.columns = ["Team", "HomeGD"]

# Away teams: negate GD to reflect their perspective
away_gd = merged_premier.groupby("AwayTeam")["GoalDifference"].sum().reset_index()
away_gd["GoalDifference"] = -away_gd["GoalDifference"]
away_gd.columns = ["Team", "AwayGD"]

# Combine both
total_gd = pd.merge(home_gd, away_gd, on="Team", how="outer").fillna(0)

# Total goal difference = home + away
total_gd["TotalGD"] = total_gd["HomeGD"] + total_gd["AwayGD"]

# Sort nicely
total_gd = total_gd.sort_values(by="TotalGD", ascending=False).reset_index(drop=True)

print(total_gd)

                Team  HomeGD  AwayGD  TotalGD
0           Man City     167     108      275
1          Liverpool     136      76      212
2            Arsenal      99      72      171
3            Chelsea      64      27       91
4          Tottenham      57      14       71
5          Newcastle      61     -16       45
6        Aston Villa      48     -14       34
7         Man United      44     -11       33
8           Brighton      19      -8       11
9          Brentford      18     -14        4
10          West Ham      11     -30      -19
11            Fulham     -13     -17      -30
12    Crystal Palace       5     -36      -31
13             Luton      -9     -24      -33
14       Bournemouth      -2     -33      -35
15     Nott'm Forest      10     -46      -36
16         West Brom     -24     -17      -41
17         Leicester      -8     -35      -43
18           Watford     -29     -14      -43
19           Ipswich     -30     -16      -46
20             Leeds     -23     -

In [15]:
total_points = pd.merge(home_points, away_points, left_on="HomeTeam", right_on="AwayTeam", how="outer").fillna(0)
total_points["Team"] = total_points["HomeTeam"].combine_first(total_points["AwayTeam"])
total_points['TotalPoints'] = total_points['HomePoints'] + total_points['AwayPoints']
total_points['HomePointsPct'] = total_points['HomePoints'] / total_points['TotalPoints']
print(total_points.head())

      HomeTeam  HomePoints     AwayTeam  AwayPoints         Team  TotalPoints  \
0      Arsenal         200      Arsenal         177      Arsenal          377   
1  Aston Villa         166  Aston Villa         129  Aston Villa          295   
2  Bournemouth          77  Bournemouth          66  Bournemouth          143   
3    Brentford         114    Brentford          86    Brentford          200   
4     Brighton         139     Brighton         124     Brighton          263   

   HomePointsPct  
0       0.530504  
1       0.562712  
2       0.538462  
3       0.570000  
4       0.528517  


In [16]:
total_points = total_points[["Team", "HomePoints", "AwayPoints", "TotalPoints", "HomePointsPct"]]
print(total_points.sort_values("HomePointsPct", ascending=False))

                Team  HomePoints  AwayPoints  TotalPoints  HomePointsPct
20  Sheffield United          26          13           39       0.666667
14             Luton          16          10           26       0.615385
19     Nott'm Forest          82          57          139       0.589928
17         Newcastle         170         121          291       0.584192
24         West Brom          15          11           26       0.576923
21       Southampton          69          51          120       0.575000
22         Tottenham         170         127          297       0.572391
3          Brentford         114          86          200       0.570000
26            Wolves         127          98          225       0.564444
1        Aston Villa         166         129          295       0.562712
25          West Ham         143         113          256       0.558594
13         Liverpool         220         174          394       0.558376
7     Crystal Palace         133         106       

In [17]:
print(total_points.columns)

Index(['Team', 'HomePoints', 'AwayPoints', 'TotalPoints', 'HomePointsPct'], dtype='object')


In [18]:
merged_premier['Shots_Difference'] = merged_premier['HS'] - merged_premier['AS']
print(merged_premier['Shots_Difference'])

0       -8
1       -4
2       16
3        0
4       -6
        ..
1895     3
1896     4
1897   -16
1898   -19
1899     5
Name: Shots_Difference, Length: 1900, dtype: int64


In [19]:
from sklearn.linear_model import LogisticRegression
merged_premier['HomeWin'] = (merged_premier['Winner'] == merged_premier['HomeTeam']).astype(int)
X = merged_premier[['Shots_Difference']]  # predictor
y = merged_premier['HomeWin']             # target

model = LogisticRegression()
model.fit(X, y)

print(f"Intercept: {model.intercept_[0]}")
print(f"Coefficient for Shots_Difference: {model.coef_[0][0]}")

# Predict probabilities
merged_premier['HomeWin_Prob'] = model.predict_proba(X)[:, 1]

Intercept: -0.47019273553972596
Coefficient for Shots_Difference: 0.07536591911688537


In [20]:
merged_premier['ShotsonTarget_Difference'] = merged_premier['HST'] - merged_premier['AST']
print(merged_premier['ShotsonTarget_Difference'])

0      -4
1      -2
2       3
3       1
4      -6
       ..
1895    0
1896    0
1897   -6
1898   -6
1899   -1
Name: ShotsonTarget_Difference, Length: 1900, dtype: int64


In [21]:
from sklearn.linear_model import LogisticRegression
merged_premier['HomeWin'] = (merged_premier['Winner'] == merged_premier['HomeTeam']).astype(int)
X = merged_premier[['ShotsonTarget_Difference']]  # predictor
y = merged_premier['HomeWin']             # target

model = LogisticRegression()
model.fit(X, y)

print(f"Intercept: {model.intercept_[0]}")
print(f"Coefficient for ShotsonTarget_Difference: {model.coef_[0][0]}")

# Predict probabilities
merged_premier['HomeWin_Prob'] = model.predict_proba(X)[:, 1]

Intercept: -0.652344044591553
Coefficient for ShotsonTarget_Difference: 0.37100346619420704


In [22]:
merged_premier['FTR'].value_counts()

FTR
H    821
A    646
D    433
Name: count, dtype: int64

In [23]:
merged_premier.groupby('Season')[['FTHG', 'FTAG']].mean().plot(marker='o')

<Axes: xlabel='Season'>

In [24]:
team_stats = pd.DataFrame({
    'AvgGoalsScored': merged_premier.groupby('HomeTeam')['FTHG'].mean(),
    'AvgGoalsConceded': merged_premier.groupby('HomeTeam')['FTAG'].mean(),
})
print(team_stats)

                  AvgGoalsScored  AvgGoalsConceded
HomeTeam                                          
Arsenal                 2.052632          1.010526
Aston Villa             1.821053          1.315789
Bournemouth             1.228070          1.263158
Brentford               1.657895          1.421053
Brighton                1.452632          1.252632
Burnley                 0.894737          1.666667
Chelsea                 1.757895          1.084211
Crystal Palace          1.357895          1.305263
Everton                 1.210526          1.273684
Fulham                  1.289474          1.460526
Ipswich                 0.736842          2.315789
Leeds                   1.280702          1.684211
Leicester               1.394737          1.500000
Liverpool               2.263158          0.831579
Luton                   1.473684          1.947368
Man City                2.684211          0.926316
Man United              1.684211          1.221053
Newcastle               1.86315

In [25]:
# Create conditions for when home or away team is leading at half time
merged_premier['HT_leader'] = merged_premier.apply(
    lambda x: 'Home' if x['HTHG'] > x['HTAG'] else ('Away' if x['HTHG'] < x['HTAG'] else 'Draw'),
    axis=1
)

# For each team, compute % of wins when leading at half time
# 1️⃣ Home teams
home_leads = merged_premier[merged_premier['HT_leader'] == 'Home'].groupby('HomeTeam').agg(
    games_leading=('HomeTeam', 'count'),
    wins_when_leading=('FTR', lambda x: (x == 'H').sum())
)
home_leads['home_win_pct_when_leading_HT'] = 100 * home_leads['wins_when_leading'] / home_leads['games_leading']

# 2️⃣ Away teams
away_leads = merged_premier[merged_premier['HT_leader'] == 'Away'].groupby('AwayTeam').agg(
    games_leading=('AwayTeam', 'count'),
    wins_when_leading=('FTR', lambda x: (x == 'A').sum())
)
away_leads['away_win_pct_when_leading_HT'] = 100 * away_leads['wins_when_leading'] / away_leads['games_leading']

# Combine into one dataframe per team
team_win_pct = pd.concat([home_leads['home_win_pct_when_leading_HT'], 
                          away_leads['away_win_pct_when_leading_HT']], axis=1).fillna(0)

team_win_pct['overall_win_pct_when_leading_HT'] = (
    (home_leads['wins_when_leading'].fillna(0) + away_leads['wins_when_leading'].fillna(0)) /
    (home_leads['games_leading'].fillna(0) + away_leads['games_leading'].fillna(0))
) * 100

# Sort descending for clarity
team_win_pct = team_win_pct.sort_values('overall_win_pct_when_leading_HT', ascending=False)

team_win_pct

,home_win_pct_when_leading_HT,away_win_pct_when_leading_HT,overall_win_pct_when_leading_HT
Liverpool,91.666667,85.294118,89.024390
Man City,87.096774,86.046512,86.666667
Arsenal,83.720930,83.720930,83.720930
Leicester,90.476190,71.428571,82.857143
Newcastle,79.487179,86.956522,82.258065
Man United,81.250000,78.260870,80.000000
Sheffield United,80.000000,75.000000,77.777778
Aston Villa,82.500000,70.370370,77.611940
Chelsea,72.222222,76.666667,74.242424
Tottenham,75.000000,69.696970,72.602740


In [26]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'vscode'

team_win_pct_sorted = team_win_pct.sort_values('overall_win_pct_when_leading_HT', ascending=False).reset_index()
team_names = team_win_pct_sorted['index']

win_fig_prem = go.Figure(data=[
    go.Bar(
        name='Home',
        x=team_names,
        y=team_win_pct_sorted['home_win_pct_when_leading_HT'],
        marker_color='royalblue',
        hovertemplate='<b>%{x}</b><br>Home Win %: %{y:.1f}%<extra></extra>'
    ),
    go.Bar(
        name='Away',
        x=team_names,
        y=team_win_pct_sorted['away_win_pct_when_leading_HT'],
        marker_color='orange',
        hovertemplate='<b>%{x}</b><br>Away Win %: %{y:.1f}%<extra></extra>'
    )
])

win_fig_prem.update_layout(
    title='PremierLeague: Home vs Away Win % When Leading at Half Time (Interactive)',
    xaxis_title='Team',
    yaxis_title='Win Percentage (%)',
    barmode='group',
    hovermode='x unified',
    template='plotly_white',
    xaxis_tickangle=-45,
    height=600
)

win_fig_prem.show()


In [27]:
# Identify who was leading at half time
merged_premier['HT_leader'] = merged_premier.apply(
    lambda x: 'Home' if x['HTHG'] > x['HTAG'] else ('Away' if x['HTHG'] < x['HTAG'] else 'Draw'),
    axis=1
)

home_losing = merged_premier[merged_premier['HT_leader'] == 'Away'].groupby('HomeTeam').agg(
    games_trailing=('HomeTeam', 'count'),
    losses_when_trailing=('FTR', lambda x: (x == 'A').sum())  # Home loss = 'A'
)
home_losing['home_loss_pct_when_losing_HT'] = (
    100 * home_losing['losses_when_trailing'] / home_losing['games_trailing']
)

away_losing = merged_premier[merged_premier['HT_leader'] == 'Home'].groupby('AwayTeam').agg(
    games_trailing=('AwayTeam', 'count'),
    losses_when_trailing=('FTR', lambda x: (x == 'H').sum())  # Away loss = 'H'
)
away_losing['away_loss_pct_when_losing_HT'] = (
    100 * away_losing['losses_when_trailing'] / away_losing['games_trailing']
)

team_loss_pct = pd.concat([
    home_losing['home_loss_pct_when_losing_HT'], 
    away_losing['away_loss_pct_when_losing_HT']
], axis=1).fillna(0)

team_loss_pct['overall_loss_pct_when_losing_HT'] = (
    (home_losing['losses_when_trailing'].fillna(0) + away_losing['losses_when_trailing'].fillna(0)) /
    (home_losing['games_trailing'].fillna(0) + away_losing['games_trailing'].fillna(0))
) * 100

team_loss_pct = team_loss_pct.sort_values('overall_loss_pct_when_losing_HT', ascending=False)

team_loss_pct.head()


,home_loss_pct_when_losing_HT,away_loss_pct_when_losing_HT,overall_loss_pct_when_losing_HT
Norwich,100.000000,100.000000,100.000000
Watford,90.909091,100.000000,95.000000
Sheffield United,94.444444,94.736842,94.594595
Southampton,90.476190,86.363636,87.692308
Luton,90.000000,75.000000,83.333333


In [28]:
import plotly.graph_objects as go
import plotly.io as pio

# Renderer setup for VS Code
pio.renderers.default = 'vscode'

# Sort for plotting
team_loss_pct_sorted = team_loss_pct.sort_values('overall_loss_pct_when_losing_HT', ascending=False).reset_index()
team_names = team_loss_pct_sorted['index']

# Create grouped bar chart
loss_fig_prem = go.Figure(data=[
    go.Bar(
        name='Home (Losing at HT)',
        x=team_names,
        y=team_loss_pct_sorted['home_loss_pct_when_losing_HT'],
        marker_color='crimson',
        hovertemplate='<b>%{x}</b><br>Home Loss %: %{y:.1f}%<extra></extra>'
    ),
    go.Bar(
        name='Away (Losing at HT)',
        x=team_names,
        y=team_loss_pct_sorted['away_loss_pct_when_losing_HT'],
        marker_color='darkorange',
        hovertemplate='<b>%{x}</b><br>Away Loss %: %{y:.1f}%<extra></extra>'
    )
])

# Layout customization
loss_fig_prem.update_layout(
    title='Premier League: Loss % When Losing at Half Time (Last 5 Seasons)',
    xaxis_title='Team',
    yaxis_title='Loss Percentage (%)',
    barmode='group',
    hovermode='x unified',
    template='plotly_white',
    xaxis_tickangle=-45,
    height=600
)

# Display the chart inline in VS Code
loss_fig_prem.show()


In [29]:
import pickle

# Assuming your figures are called win_fig and loss_fig
with open("win_fig.prem", "wb") as f:
    pickle.dump(win_fig_prem, f)

with open("loss_fig.prem", "wb") as f:
    pickle.dump(loss_fig_prem, f)

In [30]:
#shot dominance graph
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

merged_premier['ShotDiff'] = merged_premier['HS'] - merged_premier['AS']
merged_premier['ShotOnTargetDiff'] = merged_premier['HST'] - merged_premier['AST']
merged_premier['TotalShots'] = merged_premier['HS'] + merged_premier['AS']
merged_premier['HomeWin'] = merged_premier['FTR'].apply(lambda x: 1 if x == 'H' else 0)

X = merged_premier[['ShotDiff', 'ShotOnTargetDiff']]
y = merged_premier['HomeWin']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
model = LogisticRegression()
model.fit(X_scaled, y)

x_range = np.linspace(X['ShotDiff'].min(), X['ShotDiff'].max(), 200)
y_range = np.linspace(X['ShotOnTargetDiff'].min(), X['ShotOnTargetDiff'].max(), 200)
xx, yy = np.meshgrid(x_range, y_range)
grid = np.c_[xx.ravel(), yy.ravel()]
probs = model.predict_proba(scaler.transform(grid))[:, 1].reshape(xx.shape)

contour = go.Contour(
    x=x_range,
    y=y_range,
    z=probs,
    colorscale='RdYlGn',
    opacity=0.9,
    contours=dict(showlines=False),
    colorbar=dict(title='Home Win<br>Probability', tickformat='.0%'),
)

bubbles = go.Scatter(
    x=merged_premier['ShotDiff'],
    y=merged_premier['ShotOnTargetDiff'],
    mode='markers',
    marker=dict(
        size=np.sqrt(merged_premier['TotalShots']) * 1.8,  # bigger bubble for more shots
        color=merged_premier['HomeWin'],
        colorscale='RdYlGn',
        line=dict(width=1, color='white'),
        opacity=0.85,
        cmin=0,
        cmax=1,
        showscale=False
    ),
    text=[
        f"<b>{ht} vs {at}</b><br>"
        f"Shots: {hs}-{as_}<br>"
        f"On Target: {hst}-{ast}<br>"
        f"Full-time score: {fts}<br>"
        f"Result: {res}"
        for ht, at, hs, as_, hst, ast, fts, res in zip(
            merged_premier['HomeTeam'],
            merged_premier['AwayTeam'],
            merged_premier['HS'],
            merged_premier['AS'],
            merged_premier['HST'],
            merged_premier['AST'],
            merged_premier['FullTimeScore'],
            merged_premier['FTR']
        )
    ],
    hoverinfo='text'
)

boundary = go.Contour(
    x=x_range,
    y=y_range,
    z=probs,
    contours=dict(
        start=0.5, end=0.5, size=0.01, coloring='lines', showlabels=False
    ),
    showscale=False,
    line=dict(color='black', width=3, dash='dot'),
    hoverinfo='skip'
)

layout = go.Layout(
    title=dict(
        text="🏟️ <b>Match Dominance Arena:</b> Shots vs Shots on Target and Home Win Probability",
        x=0.5,
        xanchor='center'
    ),
    xaxis=dict(
        title='Shot Difference (Home − Away)',
        zeroline=True, zerolinecolor='gray', gridcolor='rgba(200,200,200,0.3)'
    ),
    yaxis=dict(
        title='Shots on Target Difference (Home − Away)',
        zeroline=True, zerolinecolor='gray', gridcolor='rgba(200,200,200,0.3)'
    ),
    paper_bgcolor='rgb(245,245,245)',
    plot_bgcolor='rgb(250,250,250)',
    hovermode='closest',
    height=700,
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig_prem_shots = go.Figure(data=[contour, boundary, bubbles], layout=layout)
fig_prem_shots.show()

with open("match_dominance_fig_prem.pkl", "wb") as f:
    pickle.dump(fig_prem_shots, f)

c:\Users\Simon\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning:

X does not have valid feature names, but StandardScaler was fitted with feature names



In [31]:
#corners
win_rate = merged_premier.groupby(['HC', 'AC'])['HomeWin'].mean().reset_index()

fig_cornersPremier = go.Figure(data=go.Heatmap(
    x=win_rate['HC'],
    y=win_rate['AC'],
    z=win_rate['HomeWin'],
    colorscale='RdYlGn',
    colorbar=dict(title="Home Win Rate", tickformat=".0%"),
))

fig_cornersPremier.update_layout(
    title="Home Win Probability by Corner Combination (Premier League)",
    xaxis_title="Home Corners (HC)",
    yaxis_title="Away Corners (AC)",
    template="plotly_white"
)

fig_cornersPremier.show()


with open("corner_heatmapPremier_fig.pkl", "wb") as f:
    pickle.dump(fig_cornersPremier, f)

In [32]:
#computing average points
merged_premier['HomePoints'] = merged_premier['FTR'].apply(lambda x: 3 if x == 'H' else (1 if x == 'D' else 0))
merged_premier['AwayPoints'] = merged_premier['FTR'].apply(lambda x: 3 if x == 'A' else (1 if x == 'D' else 0))

avg_homepoints = merged_premier.groupby('HomeTeam')['HomePoints'].mean().sort_values(ascending=False)
avg_awaypoints = merged_premier.groupby('AwayTeam')['AwayPoints'].mean().sort_values(ascending=False)
print(avg_homepoints)

HomeTeam
Man City            2.410526
Liverpool           2.315789
Arsenal             2.105263
Man United          1.800000
Newcastle           1.789474
Chelsea             1.789474
Tottenham           1.789474
Aston Villa         1.747368
West Ham            1.505263
Brentford           1.500000
Brighton            1.463158
Nott'm Forest       1.438596
Crystal Palace      1.400000
Bournemouth         1.350877
Wolves              1.336842
Everton             1.305263
Leicester           1.263158
Fulham              1.236842
Leeds               1.210526
Southampton         0.907895
Burnley             0.859649
Luton               0.842105
West Brom           0.789474
Sheffield United    0.684211
Norwich             0.631579
Watford             0.421053
Ipswich             0.368421
Name: HomePoints, dtype: float64


In [33]:
merged_premier = merged_premier.merge(avg_homepoints.rename('AvgHomePoints'), left_on='HomeTeam', right_index=True)
merged_premier = merged_premier.merge(avg_awaypoints.rename('AvgAwayPoints'), left_on='AwayTeam', right_index=True)

In [34]:
merged_premier['Date'] = pd.to_datetime(merged_premier['Date'], dayfirst=True)

def get_season(date):
    year = date.year
    if date.month >= 7:  # July or later → season starts this year
        return f"{year}/{year+1}"
    else:  # Jan-Jun → season started last year
        return f"{year-1}/{year}"

merged_premier['Season'] = merged_premier['Date'].apply(get_season)
merged_premier = merged_premier[merged_premier['Season'].isin(['2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025'])]
merged_premier = merged_premier.merge(
    merged_premier.groupby('Season').agg(
        AvgHomePointsSeason=('HomePoints', 'mean'),
        AvgAwayPointsSeason=('AwayPoints', 'mean')
    ),
    left_on='Season',
    right_index=True
)

In [35]:
#computing new table containing average points data
overall_homepoints = merged_premier.groupby('HomeTeam')['HomePoints'].mean().rename('Overall_AvgHome')
overall_awaypoints = merged_premier.groupby('AwayTeam')['AwayPoints'].mean().rename('Overall_AvgAway')

overall_total = (
    pd.concat([
        merged_premier.rename(columns={'HomeTeam' : 'Team'})[['Team', 'HomePoints']].rename(columns={'HomePoints' : 'Points'}),
        merged_premier.rename(columns={'AwayTeam' : 'Team'})[['Team', 'AwayPoints']].rename(columns={'AwayPoints' : 'Points'})
    ])
    .groupby('Team')['Points'].mean()
    .rename('Overall_AvgTotal')
) 
overall = pd.concat([overall_homepoints, overall_awaypoints, overall_total], axis=1).reset_index()
overall = overall.rename(columns={'index':'Team'})

In [36]:
#per season average points
season_home = (
    merged_premier.groupby(['Season','HomeTeam'])['HomePoints']
    .mean()
    .rename('TeamSeason_AvgHome')
    .reset_index()
    .rename(columns={'HomeTeam':'Team'})
)

season_away = (
    merged_premier.groupby(['Season','AwayTeam'])['AwayPoints']
    .mean()
    .rename('TeamSeason_AvgAway')
    .reset_index()
    .rename(columns={'AwayTeam':'Team'})
)

season_total = pd.concat([
    merged_premier.assign(Team=merged_premier['HomeTeam'], Points=merged_premier['HomePoints'])[['Season','Team','Points']],
    merged_premier.assign(Team=merged_premier['AwayTeam'], Points=merged_premier['AwayPoints'])[['Season','Team','Points']]
]).groupby(['Season','Team'])['Points'].mean().rename('TeamSeason_AvgTotal').reset_index()

team_season_df = season_home.merge(season_away, on=['Season','Team'], how='outer')
team_season_df = team_season_df.merge(season_total, on=['Season','Team'], how='outer')

team_season_wide = team_season_df.pivot_table(
    index='Team',
    columns='Season',
    values=['TeamSeason_AvgHome','TeamSeason_AvgAway','TeamSeason_AvgTotal'],
    aggfunc='first'  
)

team_season_wide.columns = [f"{metric}_{season}" for (metric, season) in team_season_wide.columns]
team_season_wide = team_season_wide.reset_index()

team_season_wide.head()


,Team,TeamSeason_AvgAway_2020/2021,TeamSeason_AvgAway_2021/2022,TeamSeason_AvgAway_2022/2023,TeamSeason_AvgAway_2023/2024,TeamSeason_AvgAway_2024/2025,TeamSeason_AvgHome_2020/2021,TeamSeason_AvgHome_2021/2022,TeamSeason_AvgHome_2022/2023,TeamSeason_AvgHome_2023/2024,TeamSeason_AvgHome_2024/2025,TeamSeason_AvgTotal_2020/2021,TeamSeason_AvgTotal_2021/2022,TeamSeason_AvgTotal_2022/2023,TeamSeason_AvgTotal_2023/2024,TeamSeason_AvgTotal_2024/2025
0,Arsenal,1.736842,1.473684,2.052632,2.210526,1.842105,1.473684,2.157895,2.368421,2.473684,2.052632,1.605263,1.815789,2.210526,2.342105,1.947368
1,Aston Villa,1.578947,1.157895,1.210526,1.473684,1.368421,1.315789,1.210526,2.000000,2.105263,2.105263,1.447368,1.184211,1.605263,1.789474,1.736842
2,Bournemouth,NaN,NaN,0.894737,1.105263,1.473684,NaN,NaN,1.157895,1.421053,1.473684,NaN,NaN,1.026316,1.263158,1.473684
3,Brentford,NaN,1.157895,1.157895,0.894737,1.315789,NaN,1.263158,1.947368,1.157895,1.631579,NaN,1.210526,1.552632,1.026316,1.473684
4,Brighton,1.052632,1.526316,1.473684,0.947368,1.526316,1.105263,1.157895,1.789474,1.578947,1.684211,1.078947,1.342105,1.631579,1.263158,1.605263


In [37]:
final_table = pd.merge(overall, team_season_wide, on='Team', how='left')
final_table = final_table.sort_values('Team').reset_index(drop=True)
final_table.fillna(0, inplace=True)
final_table.head() 

,Team,Overall_AvgHome,Overall_AvgAway,Overall_AvgTotal,TeamSeason_AvgAway_2020/2021,TeamSeason_AvgAway_2021/2022,TeamSeason_AvgAway_2022/2023,TeamSeason_AvgAway_2023/2024,TeamSeason_AvgAway_2024/2025,TeamSeason_AvgHome_2020/2021,TeamSeason_AvgHome_2021/2022,TeamSeason_AvgHome_2022/2023,TeamSeason_AvgHome_2023/2024,TeamSeason_AvgHome_2024/2025,TeamSeason_AvgTotal_2020/2021,TeamSeason_AvgTotal_2021/2022,TeamSeason_AvgTotal_2022/2023,TeamSeason_AvgTotal_2023/2024,TeamSeason_AvgTotal_2024/2025
0,Arsenal,2.105263,1.863158,1.984211,1.736842,1.473684,2.052632,2.210526,1.842105,1.473684,2.157895,2.368421,2.473684,2.052632,1.605263,1.815789,2.210526,2.342105,1.947368
1,Aston Villa,1.747368,1.357895,1.552632,1.578947,1.157895,1.210526,1.473684,1.368421,1.315789,1.210526,2.000000,2.105263,2.105263,1.447368,1.184211,1.605263,1.789474,1.736842
2,Bournemouth,1.350877,1.157895,1.254386,0.000000,0.000000,0.894737,1.105263,1.473684,0.000000,0.000000,1.157895,1.421053,1.473684,0.000000,0.000000,1.026316,1.263158,1.473684
3,Brentford,1.500000,1.131579,1.315789,0.000000,1.157895,1.157895,0.894737,1.315789,0.000000,1.263158,1.947368,1.157895,1.631579,0.000000,1.210526,1.552632,1.026316,1.473684
4,Brighton,1.463158,1.305263,1.384211,1.052632,1.526316,1.473684,0.947368,1.526316,1.105263,1.157895,1.789474,1.578947,1.684211,1.078947,1.342105,1.631579,1.263158,1.605263


In [38]:
season_home = merged_premier.groupby(['Season','HomeTeam'])['HomePoints'].mean().rename('Season_AvgHome').reset_index()
season_away = merged_premier.groupby(['Season','AwayTeam'])['AwayPoints'].mean().rename('Season_AvgAway').reset_index()

season_total = (
    pd.concat([
        merged_premier.assign(Team=merged_premier['HomeTeam'], Points=merged_premier['HomePoints'])[['Season', 'Team', 'Points']],
        merged_premier.assign(Team=merged_premier['AwayTeam'], Points=merged_premier['AwayPoints'])[['Season', 'Team', 'Points']]
    ])
    .groupby(['Season', 'Team'])['Points'].mean().rename('Season_AvgTotal').reset_index()
)

season = pd.concat([season_home, season_away, season_total], axis=1).reset_index()

season = pd.merge(season_home, season_away, left_on=['Season','HomeTeam'], right_on=['Season','AwayTeam'], how='outer')
season = pd.merge(season, season_total, left_on=['Season', 'HomeTeam'], right_on=['Season', 'Team'], how='outer')

season['Team'] = season['HomeTeam'].fillna(season['AwayTeam']).fillna(season['Team'])

season = season[['Season', 'Team', 'Season_AvgHome', 'Season_AvgAway', 'Season_AvgTotal']]

season_wide = season.pivot(index='Team', columns='Season')
season_wide.columns = [f"{season}_{metric}" for (metric, season) in season_wide.columns]
season_wide = season_wide.reset_index()

final_table = pd.merge(overall, season_wide, on='Team', how='left')
print(final_table)

                Team  Overall_AvgHome  Overall_AvgAway  Overall_AvgTotal  \
0            Arsenal         2.105263         1.863158          1.984211   
1        Aston Villa         1.747368         1.357895          1.552632   
2        Bournemouth         1.350877         1.157895          1.254386   
3          Brentford         1.500000         1.131579          1.315789   
4           Brighton         1.463158         1.305263          1.384211   
5            Burnley         0.859649         0.859649          0.859649   
6            Chelsea         1.789474         1.547368          1.668421   
7     Crystal Palace         1.400000         1.115789          1.257895   
8            Everton         1.305263         1.115789          1.210526   
9             Fulham         1.236842         1.144737          1.190789   
10           Ipswich         0.368421         0.789474          0.578947   
11             Leeds         1.210526         1.035088          1.122807   
12         L

In [39]:
season_home = season_home.rename(columns={'HomeTeam':'Team'})
season_away = season_away.rename(columns={'AwayTeam':'Team'})

season_ha = pd.merge(season_home, season_away, on=['Season','Team'], how='outer')
season = pd.merge(season_ha, season_total, on=['Season','Team'], how='outer')

season_wide = season.pivot(index='Team', columns='Season')
season_wide.columns = [f"{metric}_{season}" for (metric, season) in season_wide.columns]
season_wide = season_wide.reset_index()
final_table = pd.merge(overall, season_wide, on='Team', how='left')
final_table = final_table.fillna(0)
print(final_table)
final_table.to_excel('Premier_Team_AvgPoints.xlsx', index=False)

                Team  Overall_AvgHome  Overall_AvgAway  Overall_AvgTotal  \
0            Arsenal         2.105263         1.863158          1.984211   
1        Aston Villa         1.747368         1.357895          1.552632   
2        Bournemouth         1.350877         1.157895          1.254386   
3          Brentford         1.500000         1.131579          1.315789   
4           Brighton         1.463158         1.305263          1.384211   
5            Burnley         0.859649         0.859649          0.859649   
6            Chelsea         1.789474         1.547368          1.668421   
7     Crystal Palace         1.400000         1.115789          1.257895   
8            Everton         1.305263         1.115789          1.210526   
9             Fulham         1.236842         1.144737          1.190789   
10           Ipswich         0.368421         0.789474          0.578947   
11             Leeds         1.210526         1.035088          1.122807   
12         L

In [40]:
#computing weighted probabilities of winning, assuming that 50% of probability is determined by overall performance, and 50% by season performance
seasons = ['2020/2021','2021/2022','2022/2023','2023/2024','2024/2025']

for season in seasons:
    final_table[f'WeightedScore_{season}'] = (
        0.5 * final_table['Overall_AvgTotal'] +
        0.5 * final_table[f'Season_AvgTotal_{season}']
    )
    final_table[f'WinProb_{season}'] = (
        final_table[f'WeightedScore_{season}'] / final_table[f'WeightedScore_{season}'].sum()
    )


In [41]:
season = '2024/2025'

# Weighted total points per game
final_table['WeightedPPG'] = 0.5 * final_table['Overall_AvgTotal'] + 0.5 * final_table[f'Season_AvgTotal_{season}']

# Merge home team strength
merged_premier = merged_premier.merge(
    final_table[['Team', 'WeightedPPG']],
    left_on='HomeTeam',
    right_on='Team',
    how='left'
).rename(columns={'WeightedPPG':'HomeStrength'}).drop(columns='Team')

# Merge away team strength
merged_premier = merged_premier.merge(
    final_table[['Team', 'WeightedPPG']],
    left_on='AwayTeam',
    right_on='Team',
    how='left'
).rename(columns={'WeightedPPG':'AwayStrength'}).drop(columns='Team')


In [42]:
merged_premier['ExpHome'] = merged_premier['HomeStrength'] / (merged_premier['HomeStrength'] + merged_premier['AwayStrength'])
merged_premier['ExpAway'] = merged_premier['AwayStrength'] / (merged_premier['HomeStrength'] + merged_premier['AwayStrength'])

draw_prob_overall = (merged_premier['FTR'] == 'D').mean()
home_draw_prob = merged_premier.groupby('HomeTeam').apply(lambda x: (x['FTR'] == 'D').mean())
away_draw_prob = merged_premier.groupby('AwayTeam').apply(lambda x: (x['FTR'] == 'D').mean())

merged_premier = merged_premier.merge(home_draw_prob.rename('HomeDrawProb'), left_on='HomeTeam', right_index=True)
merged_premier = merged_premier.merge(away_draw_prob.rename('AwayDrawProb'), left_on='AwayTeam', right_index=True)
merged_premier['draw_prob'] = (merged_premier['HomeDrawProb'] + merged_premier['AwayDrawProb']) / 2
merged_premier['HomeProb'] = merged_premier['ExpHome'] * (1 - merged_premier['draw_prob'])
merged_premier['AwayProb'] = merged_premier['ExpAway'] * (1 - merged_premier['draw_prob'])


C:\Users\Simon\AppData\Local\Temp\ipykernel_24304\2056345844.py:5: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

C:\Users\Simon\AppData\Local\Temp\ipykernel_24304\2056345844.py:6: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [43]:
merged_premier['1XBet_HomeProb'] = 1 / merged_premier['1XBH']
merged_premier['1XBet_DrawProb'] = 1 / merged_premier['1XBD']
merged_premier['1XBet_AwayProb'] = 1 / merged_premier['1XBA']

merged_premier['B365_HomeProb'] = 1 / merged_premier['B365H']
merged_premier['B365_DrawProb'] = 1 / merged_premier['B365D']
merged_premier['B365_AwayProb'] = 1 / merged_premier['B365A']

In [44]:
merged_premier['Diff_Home_1XBet'] = merged_premier['HomeProb'] - merged_premier['1XBet_HomeProb']
merged_premier['Diff_Draw_1XBet'] = merged_premier['draw_prob'] - merged_premier['1XBet_DrawProb']
merged_premier['Diff_Away_1XBet'] = merged_premier['AwayProb'] - merged_premier['1XBet_AwayProb']

# Difference between model and Bet365
merged_premier['Diff_Home_B365'] = merged_premier['HomeProb'] - merged_premier['B365_HomeProb']
merged_premier['Diff_Draw_B365'] = merged_premier['draw_prob'] - merged_premier['B365_DrawProb']
merged_premier['Diff_Away_B365'] = merged_premier['AwayProb'] - merged_premier['B365_AwayProb']

In [45]:
merged_premier_betting = merged_premier.dropna(subset=['1XBH', 'B365H']).copy()

In [46]:
def decide_bet(row, prefix):
    # model probabilities
    probs = [row['HomeProb'], row['draw_prob'], row['AwayProb']]
    outcomes = ['H', 'D', 'A']
    
    # identify highest probability
    sorted_idx = sorted(range(3), key=lambda i: probs[i], reverse=True)
    
    # Strategy 1: bet only on top outcome if > 0.7
    if probs[sorted_idx[0]] > 0.7:
        return [outcomes[sorted_idx[0]]]
    
    # Strategy 2: bet on top 2 outcomes if combined > 0.8
    elif probs[sorted_idx[0]] + probs[sorted_idx[1]] > 0.8:
        return [outcomes[sorted_idx[0]], outcomes[sorted_idx[1]]]
    
    # Strategy 3: skip match
    else:
        return []

merged_premier_betting['BetSignal'] = merged_premier_betting.apply(decide_bet, axis=1, prefix='model')


In [47]:
def expected_return(row, bookmaker):
    if bookmaker == '1XBet':
        odds_map = {
            'H': row['1XBH'],
            'D': row['1XBD'],
            'A': row['1XBA']
        }
    elif bookmaker == 'B365':
        odds_map = {
            'H': row['B365H'],
            'D': row['B365D'],
            'A': row['B365A']
        }
    model_map = {
        'H': row['HomeProb'],
        'D': row['draw_prob'],
        'A': row['AwayProb']
    }
    returns = [model_map[outcome]*odds_map[outcome]-1 for outcome in row['BetSignal']]
    return max(returns) if returns else 0

merged_premier_betting['ExpReturn_1XBet'] = merged_premier_betting.apply(lambda x: expected_return(x, '1XBet'), axis=1)
merged_premier_betting['ExpReturn_B365'] = merged_premier_betting.apply(lambda x: expected_return(x, 'B365'), axis=1)


In [48]:
stake = 10

def compute_returns(merged_premier_betting, book_prefix):
    odds_map = {
        "1XBet": "1XB",
        "B365": "B365"
    }
    odds_prefix = odds_map[book_prefix]
    outcomes = {
        "Home": ("HomeProb",  f"{odds_prefix}H", f"Diff_Home_{book_prefix}"),
        "Draw": ("draw_prob", f"{odds_prefix}D", f"Diff_Draw_{book_prefix}"),
        "Away": ("AwayProb",  f"{odds_prefix}A", f"Diff_Away_{book_prefix}")
    }

    for outcome, (model_col, odds_col, diff_col) in outcomes.items():

        merged_premier_betting[f"Bet_{outcome}_{book_prefix}"] = merged_premier_betting[diff_col] > 0

        merged_premier_betting[f"Profit_{outcome}_{book_prefix}"] = merged_premier_betting.apply(
            lambda row:
                (stake * (row[odds_col] - 1))            
                if (row["FTR"] == outcome[0] and row[f"Bet_{outcome}_{book_prefix}"])
                else (-stake)                            
                if row[f"Bet_{outcome}_{book_prefix}"]
                else 0,                                   
            axis=1
        )
    merged_premier_betting[f"TotalProfit_{book_prefix}"] = (
        merged_premier_betting[f"Profit_Home_{book_prefix}"] +
        merged_premier_betting[f"Profit_Draw_{book_prefix}"] +
        merged_premier_betting[f"Profit_Away_{book_prefix}"]
    )

    return merged_premier_betting

merged_premier_betting = compute_returns(merged_premier_betting, "1XBet")
merged_premier_betting = compute_returns(merged_premier_betting, "B365")

total_profit_1XBet = merged_premier_betting["TotalProfit_1XBet"].sum()
total_profit_B365 = merged_premier_betting["TotalProfit_B365"].sum()

total_bets_1XBet = (
    merged_premier_betting["Bet_Home_1XBet"].sum() +
    merged_premier_betting["Bet_Draw_1XBet"].sum() +
    merged_premier_betting["Bet_Away_1XBet"].sum()
)

total_bets_B365 = (
    merged_premier_betting["Bet_Home_B365"].sum() +
    merged_premier_betting["Bet_Draw_B365"].sum() +
    merged_premier_betting["Bet_Away_B365"].sum()
)

roi_1xbet = total_profit_1XBet / (total_bets_1XBet * stake) if total_bets_1XBet > 0 else 0
roi_b365  = total_profit_B365 / (total_bets_B365 * stake) if total_bets_B365 > 0 else 0


print("1XBet Performance")
print(f"Total Profit: €{total_profit_1XBet:.2f}")
print(f"Total Bets Placed: {total_bets_1XBet}")
print(f"ROI: {roi_1xbet:.2%}\n")

print("Bet365 Performance")
print(f"Total Profit: €{total_profit_B365:.2f}")
print(f"Total Bets Placed: {total_bets_B365}")
print(f"ROI: {roi_b365:.2%}\n")


1XBet Performance
Total Profit: €607.60
Total Bets Placed: 547
ROI: 11.11%

Bet365 Performance
Total Profit: €309.80
Total Bets Placed: 511
ROI: 6.06%



In [49]:
merged_premier_betting["Cumulative_1XBet"] = merged_premier_betting["TotalProfit_1XBet"].cumsum()
merged_premier_betting["Cumulative_B365"] = merged_premier_betting["TotalProfit_B365"].cumsum()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=merged_premier_betting.index,
    y=merged_premier_betting["Cumulative_1XBet"],
    mode="lines",
    name="1XBet Cumulative Profit"
))

fig.add_trace(go.Scatter(
    x=merged_premier_betting.index,
    y=merged_premier_betting["Cumulative_B365"],
    mode="lines",
    name="Bet365 Cumulative Profit"
))

fig.update_layout(
    title="Cumulative Profit Over Time",
    xaxis_title="Match Index",
    yaxis_title="Profit (€)",
    hovermode="x unified"
)

fig.show()


In [51]:
model_p = merged_premier_betting["HomeProb"]
book_p = merged_premier_betting["1XBet_HomeProb"]
edge = merged_premier_betting["Diff_Home_1XBet"]

bins = 40
hist, xedges, yedges = np.histogram2d(model_p, book_p, bins=bins, weights=edge)
fig_premier_betting = go.Figure()

fig_premier_betting.add_trace(go.Heatmap(
    x=xedges,
    y=yedges,
    z=hist.T,     
    colorscale="RdBu", 
    colorbar=dict(title="Model Edge")
))

bet_mask = merged_premier_betting["Bet_Home_1XBet"] == True

fig_premier_betting.add_trace(go.Scatter(
    x=model_p[bet_mask],
    y=book_p[bet_mask],
    mode="markers",
    marker=dict(size=4, color="black"),
    name="Bets Taken"
))

fig_premier_betting.update_layout(
    title="Model vs Bookmaker — Probability Comparison and Betting Signals",
    xaxis_title="Model Home Probability",
    yaxis_title="Bookmaker Home Probability",
    template="plotly_white"
)
fig_premier_betting.show()

with open("model_vs_bookmaker_premier_plot.pkl", "wb") as f:
    pickle.dump(fig_premier_betting, f)